## Ejercicio de Evaluación: Identificación Automática de Idioma

### Introducción al Problema

La **identificación automática de idioma** (*Language Identification*, LID) es una tarea clásica del Procesamiento del Lenguaje Natural que consiste en determinar, a partir de un fragmento de texto, en qué lengua está escrito. Constituye habitualmente el primer paso de muchos sistemas: enrutado de mensajes en atención al cliente multilingüe, indexado de documentos web, traducción automática, filtrado de contenido, etc.

A pesar de parecer un problema sencillo, la LID presenta retos relevantes cuando los textos son **breves**, cuando aparecen **palabras no vistas** durante el entrenamiento, o cuando se trata de **lenguas próximas** que comparten vocabulario y morfología (caso característico de las lenguas romances).

Este ejercicio aborda el problema desde **los tres paradigmas estudiados en el curso**:
- **Modelos estadísticos** (Unidad 3): n-gramas como modelo de lenguaje.
- **Modelos vectoriales** (Unidad 4): TF-IDF combinado con un clasificador supervisado.
- **Modelos neuronales** (Unidad 5): redes neuronales sobre representaciones del texto.


### Dataset de trabajo

Para este ejercicio dispone de **dos ficheros** que constituyen el conjunto de trabajo:

- `dataset-idiomas-train.csv`: corpus de entrenamiento.
- `dataset-idiomas-nuevos.csv`: conjunto de **mensajes nuevos** recibidos posteriormente, que se utilizará para la evaluación final del sistema.

Se trata de un dataset sintético controlado, diseñado específicamente para esta práctica.

#### Estructura del Dataset
- **Formato**: CSV con tres columnas.
- **Columnas**:
  - `identificador`: ID único para cada texto.
  - `idioma`: etiqueta del idioma (variable objetivo).
  - `texto`: contenido textual a clasificar.

#### Idiomas presentes
El dataset incluye textos en cinco idiomas, codificados como:
- `es` — español
- `pt` — portugués
- `it` — italiano
- `fr` — francés
- `en` — inglés

#### Características del problema
- Los textos son **cortos**, de longitud variable (desde una sola palabra hasta varias frases concatenadas).
- En el conjunto de mensajes nuevos hay una proporción significativa de **vocabulario no visto** durante el entrenamiento.
- Tres de los cinco idiomas (es, pt, it) son lenguas **romances próximas**, con vocabulario y morfología solapados.
- El texto puede incluir **mayúsculas expresivas** y **signos de puntuación repetidos**.


In [1]:
# Importamos las bibliotecas necesarias
import os
import pandas as pd
import re
import string
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Definimos las rutas
DATA_DIR = "dataset"
dataset_path = os.path.join(DATA_DIR, "dataset-idiomas-train.csv")

## Cargamos los datasets:

In [2]:
# Función para cargar el dataset
def cargar_dataset(ruta=dataset_path, crear_sintético_si_falta=True):
    """
    Carga el dataset
    """
    if os.path.exists(ruta):
        # Cargamos el dataset real
        df = pd.read_csv(ruta)
        print(f"Dataset real cargado: {ruta}")
        
        # Verificamos estructura
        print(f"Dimensiones del dataset: {df.shape}")
        print(f"Columnas disponibles: {', '.join(df.columns)}")
        
        # Limpiamos columnas innecesarias si existen
        if 'Unnamed: 0' in df.columns:
            df = df.drop('Unnamed: 0', axis=1)
        
        # Renombramos columnas para consistencia si es necesario
        if 'texto' in df.columns:
            df = df.rename(columns={'texto': 'text'})
        if 'idioma' in df.columns:
            df = df.rename(columns={'idioma': 'idioma'})
        
       
    else:
        raise FileNotFoundError(f"El archivo {ruta} no existe")
    
    print(f"Dataset preparado: {len(df)} ejemplos con {df['idioma'].nunique()} idiomas diferentes")
    
    return df

In [3]:
try:
    df = cargar_dataset()
    
    # Información general sobre el dataset
    print("\nInformación general del dataset:")
    print(f"- Número de idiomas únicos: {df['idioma'].nunique()}")
    print(f"- Idiomas presentes: {', '.join(sorted(df['idioma'].unique()))}\n\n")

    # ruta de nuevos:
    ruta_nuevos = os.path.join(DATA_DIR, "dataset-idiomas-nuevos.csv")
    df_nuevos = cargar_dataset(ruta = ruta_nuevos)
    
except Exception as e:
    print(f"Error al cargar el dataset: {e}")
    print("Continuaremos la práctica con datos de ejemplo")
    df = None

Dataset real cargado: dataset\dataset-idiomas-train.csv
Dimensiones del dataset: (735, 3)
Columnas disponibles: identificador, idioma, texto
Dataset preparado: 735 ejemplos con 5 idiomas diferentes

Información general del dataset:
- Número de idiomas únicos: 5
- Idiomas presentes: en, es, fr, it, pt


Dataset real cargado: dataset\dataset-idiomas-nuevos.csv
Dimensiones del dataset: (165, 3)
Columnas disponibles: identificador, idioma, texto
Dataset preparado: 165 ejemplos con 5 idiomas diferentes


## Analizamos brevemente las características y distribución de las clases:

In [4]:
# Análisis de distribución de idiomas
def analizar_distribucion_idiomas(df):
    """
    Analiza la distribución de idiomas en el dataset.
    
    Args:
        df: DataFrame con columna 'idioma'
        
    Returns:
        Estadísticas de la distribución de idiomas
    """
    # Contamos idiomas
    idioma_counts = df['idioma'].value_counts()
    idioma_percentages = df['idioma'].value_counts(normalize=True) * 100
    
    # Creamos DataFrame con información
    idioma_stats = pd.DataFrame({
        'Cantidad': idioma_counts,
        'Porcentaje': idioma_percentages
    })
    
    # Calculamos estadísticas adicionales
    stats = {
        'total_idiomas': len(idioma_counts),
        'media_ejemplos': idioma_counts.mean(),
        'min_ejemplos': idioma_counts.min(),
        'max_ejemplos': idioma_counts.max(),
        'balance_ratio': idioma_counts.min() / idioma_counts.max(),
        'desviacion_estandar': idioma_counts.std()
    }
    
    # Analizamos balance
    print("Análisis de balance del dataset:")
    print(f"- Total de idiomas: {stats['total_idiomas']}")
    print(f"- Media de ejemplos por idioma: {stats['media_ejemplos']:.1f}")
    print(f"- Mínimo de ejemplos: {stats['min_ejemplos']} (idioma: {idioma_counts.idxmin()})")
    print(f"- Máximo de ejemplos: {stats['max_ejemplos']} (idioma: {idioma_counts.idxmax()})")
    print(f"- Ratio de balance (min/max): {stats['balance_ratio']:.3f}")
    print(f"- Coeficiente de variación: {stats['desviacion_estandar']/stats['media_ejemplos']:.3f}")
    
    # Evaluamos si el dataset está balanceado
    if stats['balance_ratio'] > 0.8:
        print("\n✓ El dataset está bien balanceado entre los diferentes idiomas.")
    elif stats['balance_ratio'] > 0.5:
        print("\n⚠ El dataset está moderadamente balanceado. Podría beneficiarse de técnicas de balanceo.")
    else:
        print("\n⚠ El dataset está desbalanceado. Considerar técnicas como sobremuestreo o submuestreo.")
    
    return idioma_stats, stats

In [5]:
# Analizamos distribución de idiomas
if df is not None:
    idioma_stats, idioma_balance_stats = analizar_distribucion_idiomas(df)
    
    # Mostramos tabla de distribución
    print("\nDistribución detallada de idiomas:")
    display(idioma_stats.sort_values(by='Cantidad', ascending=False))

Análisis de balance del dataset:
- Total de idiomas: 5
- Media de ejemplos por idioma: 147.0
- Mínimo de ejemplos: 147 (idioma: it)
- Máximo de ejemplos: 147 (idioma: it)
- Ratio de balance (min/max): 1.000
- Coeficiente de variación: 0.000

✓ El dataset está bien balanceado entre los diferentes idiomas.

Distribución detallada de idiomas:


,Cantidad,Porcentaje
idioma,,
it,147,20.0
es,147,20.0
pt,147,20.0
fr,147,20.0
en,147,20.0


### Análisis de Ejemplos por idioma

In [6]:
# Función para mostrar ejemplos representativos de cada idioma
def mostrar_ejemplos_por_idioma(df, n_ejemplos=3):
    """
    Muestra ejemplos representativos para cada idioma.
    
    Args:
        df: DataFrame con textos e idiomas
        n_ejemplos: Número de ejemplos a mostrar por idioma
    """
    # Para cada idioma, mostramos n ejemplos
    for idioma in sorted(df['idioma'].unique()):
        print(f"\n{idioma}:")
        print("-" * 50)
        
        # Obtenemos ejemplos de diferentes longitudes para mayor representatividad
        idioma_df = df[df['idioma'] == idioma].copy()
        
        ejemplos = idioma_df.sample(min(n_ejemplos, len(idioma_df))).reset_index(drop=True)
        
        # Mostramos los ejemplos
        for i, ejemplo in enumerate(ejemplos['text']):
            print(f"{i+1}. \"{ejemplo}\"")
            
    print("\nEstos ejemplos nos ayudan a entender la variabilidad lingüística en cada idioma.")

In [7]:
# Mostramos ejemplos representativos
if df is not None:
    mostrar_ejemplos_por_idioma(df)


en:
--------------------------------------------------
1. "I want to book now."
2. "My name is Ana."
3. "I have to sleep this week."

es:
--------------------------------------------------
1. "¿Puedo llamar aquí?"
2. "Hola."
3. "No hablo bien el idioma. ¿Cuánto cuesta esto?"

fr:
--------------------------------------------------
1. "Je ne parle pas bien la langue."
2. "J'aimerais étudier bientôt."
3. "Bonjour."

it:
--------------------------------------------------
1. "Grazie. Buongiorno."
2. "Mi dispiace!!"
3. "Ho molta fame."

pt:
--------------------------------------------------
1. "Tchau."
2. "Não falo bem o idioma."
3. "Preciso pagar hoje."

Estos ejemplos nos ayudan a entender la variabilidad lingüística en cada idioma.


## Preprocesado del dataset:
Antes de empezar con el tf-idf vamos a hacer un poco de limpieza del dataset para mejorar el rendimiento del modelo.

In [8]:
def convertir_minusculas(texto):
    """
    Convierte todo el texto a minúsculas.
    
    Args:
        texto (str): Texto a procesar
    
    Returns:
        str: Texto en minúsculas
    """
    return texto.lower()

def eliminar_puntuacion(texto):
    """
    Elimina signos de puntuación del texto.
    
    Args:
        texto (str): Texto a procesar
    
    Returns:
        str: Texto sin signos de puntuación
    """
    # Utilizamos una traducción para eliminar toda la puntuación
    translator = str.maketrans('', '', string.punctuation)
    return texto.translate(translator)

def eliminar_numeros(texto):
    """
    Elimina números del texto.
    
    Args:
        texto (str): Texto a procesar
    
    Returns:
        str: Texto sin números
    """
    return re.sub(r'\d+', '', texto)

def eliminar_espacios_extra(texto):
    """
    Elimina espacios en blanco múltiples y al principio/final.
    
    Args:
        texto (str): Texto a procesar
    
    Returns:
        str: Texto con espacios normalizados
    """
    return re.sub(r'\s+', ' ', texto).strip()

In [9]:
def preprocesar_texto(texto, opciones=None):
    """
    Aplica un pipeline completo de preprocesamiento a un texto.
    
    Args:
        texto (str): Texto a procesar
        opciones (dict): Diccionario con opciones de preprocesamiento
            - minusculas (bool): Convertir a minúsculas
            - puntuacion (bool): Eliminar puntuación
            - numeros (bool): Eliminar números
            - stopwords (bool): Eliminar stopwords
            - stemming (bool): Aplicar stemming
            - lematizacion (bool): Aplicar lematización
    
    Returns:
        str or list: Texto preprocesado (como string o lista de tokens según la configuración)
    """
    # Opciones por defecto
    if opciones is None:
        opciones = {
            'minusculas': True,
            'puntuacion': True,
            'numeros': True
        }
    
    # Aplicamos transformaciones básicas
    if opciones.get('minusculas', True):
        texto = convertir_minusculas(texto)
    
    if opciones.get('puntuacion', True):
        texto = eliminar_puntuacion(texto)
    
    if opciones.get('numeros', True):
        texto = eliminar_numeros(texto)
    
    texto_limpio = eliminar_espacios_extra(texto)
    
    return texto_limpio

In [10]:
def preprocesar_dataset(df, config_preprocesamiento):
    """
    Aplica preprocesamiento a todo un dataset.
    
    Args:
        df (DataFrame): DataFrame con textos a procesar en la columna 'Text'
        config_preprocesamiento (dict): Configuración de preprocesamiento
    
    Returns:
        DataFrame: DataFrame con textos preprocesados en la columna 'Text_Procesado'
    """
    # Creamos una copia para no modificar el original
    df_procesado = df.copy()
    
    # Inicializamos la columna de textos procesados
    df_procesado['Text_Procesado'] = ""

    # Procesamos cada texto
    for i, row in df_procesado.iterrows():
        df_procesado.at[i, 'Text_Procesado'] = preprocesar_texto(row['text'], config_preprocesamiento)
        
        # Mostramos progreso cada 100 documentos
        if (i+1) % 100 == 0:
            print(f"Procesados {i+1}/{len(df)} documentos ({(i+1)/len(df)*100:.1f}%)")
    
    return df_procesado

In [11]:
config_final = {
    'minusculas': True,
    'puntuacion': True,
    'numeros': True}

# Aplicamos preprocesamiento al conjunto de entrenamiento
df = preprocesar_dataset(df, config_final)

Procesados 100/735 documentos (13.6%)
Procesados 200/735 documentos (27.2%)
Procesados 300/735 documentos (40.8%)
Procesados 400/735 documentos (54.4%)
Procesados 500/735 documentos (68.0%)
Procesados 600/735 documentos (81.6%)
Procesados 700/735 documentos (95.2%)


## Dividimos en entrenamiento y validación:

Lo hacemos así para entrenar posteriormente 3 modelos y ver cuál funciona mejor sin mirar el conjunto de test.

In [12]:
# Dividimos el dataset de entrenamiento en entrenamiento (80%) y validación (20%)
# Usamos 'stratify' para asegurar que ambos conjuntos tengan la misma proporción de idiomas
X_train, X_val, y_train, y_val = train_test_split(
    df['Text_Procesado'], 
    df['idioma'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['idioma']
)

print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de validación: {len(X_val)}")

Muestras de entrenamiento: 588
Muestras de validación: 147


## Matriz de características
Dado que en el reconocimiento de idiomas, las palabras fuera de vocabulario son frecuentes, vamos a usar un modelo de n-gramas (desde 3 a 5) para resolver este problema. Además lo vamos a hacer a nivel de caracteres por lo mismo. La elección de 3 a 5 se ha hecho ya que por ser lenguas romances la mayoría comparten bigramas y con esos intervalos da mejores resultados. Además usamos el smooth_idf para usa el suavizado de Laplace en el IDF.

In [13]:
vectorizador_tfidf = TfidfVectorizer(
    analyzer='char',           
    ngram_range=(3, 5),
    max_features=10000,
    sublinear_tf=True,         
    smooth_idf=True,
    min_df=2
)

# Ajustamos con entrenamiento y transformamos ambos conjuntos
X_train_tfidf = vectorizador_tfidf.fit_transform(X_train)
X_val_tfidf = vectorizador_tfidf.transform(X_val)


print(f"Vocabulario de n-gramas generado: {X_train_tfidf.shape[1]} subcadenas.")

Vocabulario de n-gramas generado: 5651 subcadenas.


In [14]:
# Definimos un diccionario con los modelos a evaluar
modelos = {
    "Naive Bayes Multinomial": MultinomialNB(alpha=0.1),
    "Regresión Logística": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Linear SVM": LinearSVC(C=1.0, random_state=42)
}

# Entrenamiento y evaluación en el set de validación
resultados = {}

for nombre, modelo in modelos.items():
    print(f"Entrenando {nombre}...")
    modelo.fit(X_train_tfidf, y_train)
    
    # Predicciones
    predicciones = modelo.predict(X_val_tfidf)
    
    # Métrica
    acc = accuracy_score(y_val, predicciones)
    resultados[nombre] = {
        'modelo_objeto': modelo,
        'accuracy': acc,
        'reporte': classification_report(y_val, predicciones)
    }
    print(f"-> Accuracy en validación: {acc:.4f}\n")

Entrenando Naive Bayes Multinomial...
-> Accuracy en validación: 0.9728

Entrenando Regresión Logística...
-> Accuracy en validación: 0.9796

Entrenando Linear SVM...
-> Accuracy en validación: 0.9796



In [15]:
# Buscamos el modelo con mayor Accuracy
mejor_nombre = max(resultados, key=lambda k: resultados[k]['accuracy'])
mejor_modelo = resultados[mejor_nombre]['modelo_objeto']

print(f" El mejor modelo es: {mejor_nombre}")
print(f" Accuracy alcanzado: {resultados[mejor_nombre]['accuracy']:.4f}")

print("\nReporte de Clasificación Detallado sobre el conjunto de validación:\n")
print(resultados[mejor_nombre]['reporte'])

 El mejor modelo es: Regresión Logística
 Accuracy alcanzado: 0.9796

Reporte de Clasificación Detallado sobre el conjunto de validación:

              precision    recall  f1-score   support

          en       0.91      1.00      0.95        30
          es       1.00      1.00      1.00        29
          fr       1.00      0.90      0.95        29
          it       1.00      1.00      1.00        30
          pt       1.00      1.00      1.00        29

    accuracy                           0.98       147
   macro avg       0.98      0.98      0.98       147
weighted avg       0.98      0.98      0.98       147



## Resultados en Test

Finalmente vamos a ver los resultados sobre el conjunto de test del mejor modelo previo.

In [16]:
# Preprocesamos el dataframe de mensajes nuevos usando la función anterior:
print("Preprocesando el conjunto de mensajes nuevos...")
df_nuevos_procesado = preprocesar_dataset(df_nuevos, config_final)

# Vecotorizamos los nuevos textos con el TF-IDF ya entrenado
X_nuevos_tfidf = vectorizador_tfidf.transform(df_nuevos_procesado['Text_Procesado'])

# Predecir con el mejor modelo
predicciones_nuevos = mejor_modelo.predict(X_nuevos_tfidf)

# Añadir las predicciones al DataFrame para guardarlo o analizarlo
df_nuevos_procesado['idioma_predicho'] = predicciones_nuevos

# Calcular el rendimiento final en los nuevos datos
print("\nEvaluación Final en 'dataset-idiomas-nuevos.csv':")
print(classification_report(df_nuevos_procesado['idioma'], df_nuevos_procesado['idioma_predicho']))

Preprocesando el conjunto de mensajes nuevos...
Procesados 100/165 documentos (60.6%)

Evaluación Final en 'dataset-idiomas-nuevos.csv':
              precision    recall  f1-score   support

          en       0.94      0.94      0.94        33
          es       0.90      0.79      0.84        33
          fr       0.93      0.85      0.89        33
          it       0.97      0.91      0.94        33
          pt       0.76      0.97      0.85        33

    accuracy                           0.89       165
   macro avg       0.90      0.89      0.89       165
weighted avg       0.90      0.89      0.89       165



Ahora exportamos el modelo para usarlo en el pipeline simulado.

In [17]:
import os
import pickle

# Definimos los nombres de los modelos:
ruta_vectorizador = "vectorizador_tfidf_T1.pkl"
ruta_clasificador = "mejor_modelo_T1.pkl"

# Guardar el vectorizador TF-IDF:
with open(ruta_vectorizador, 'wb') as f:
    pickle.dump(vectorizador_tfidf, f)

# Guardar el modelo clasificador:
with open(ruta_clasificador, 'wb') as f:
    pickle.dump(mejor_modelo, f)

print("¡Artefactos exportados exitosamente")

¡Artefactos exportados exitosamente


## Visualización de ejemplos

Veamos los ejemplos de test sobre los que se equivoca:

In [22]:
# Identificamos las filas donde el modelo ha fallado en el dataset de test
errores = df_nuevos_procesado[df_nuevos_procesado['idioma'] != df_nuevos_procesado['idioma_predicho']]

print(f"Total de errores en el conjunto de test: {len(errores)} de {len(df_nuevos_procesado)}")

# Mostramos los ejemplos de errores para auditoría
if len(errores) > 0:
    print("\nDado que son solo 18 errores, vamos a mostralos:")
    columnas_mostrar = ['text', 'idioma', 'idioma_predicho']
    display(errores[columnas_mostrar])
else:
    print("No se encontraron errores en el conjunto de test.")

Total de errores en el conjunto de test: 18 de 165

Dado que son solo 18 errores, vamos a mostralos:


,text,idioma,idioma_predicho
6,Danser.,fr,pt
16,Nadar.,es,pt
40,Sta piovendo da stamattina.,it,pt
46,Chanter.,fr,pt
49,Manger.,fr,it
50,Reservar.,pt,es
65,Rest.,en,es
75,Danser.,fr,pt
77,Enviar.,es,pt
90,Nadar.,es,pt


Son ejemplos casi todos de una palabra, normal que en estos casos no tenga datos para predecir correctamente. Vamos a probar con algo más realista de lo que nos encontraremos en el foro:

In [23]:
import pandas as pd

# Creamos un dataset sintético con 30 ejemplos (6 por idioma)
datos_prueba = [
    # Español (es)
    {'text': 'Hola, ¿cómo estás hoy?', 'idioma': 'es'},
    {'text': 'El sistema de detección de idiomas funciona bien.', 'idioma': 'es'},
    {'text': 'Mañana iré a caminar por el parque central.', 'idioma': 'es'},
    {'text': 'Me gusta mucho la comida mediterránea.', 'idioma': 'es'},
    {'text': '¿Podrías enviarme el reporte final por correo?', 'idioma': 'es'},
    {'text': 'La inteligencia artificial está cambiando el mundo.', 'idioma': 'es'},
    
    # Portugués (pt)
    {'text': 'Olá, como você está hoje?', 'idioma': 'pt'},
    {'text': 'Eu gosto muito de viajar para a praia no verão.', 'idioma': 'pt'},
    {'text': 'O café da manhã está servido na mesa.', 'idioma': 'pt'},
    {'text': 'Esta é uma frase curta em português.', 'idioma': 'pt'},
    {'text': 'Muito obrigado pela sua ajuda com este projeto.', 'idioma': 'pt'},
    {'text': 'Amanhã vamos ao cinema ver um filme novo.', 'idioma': 'pt'},

    # Italiano (it)
    {'text': 'Buongiorno, come va la giornata?', 'idioma': 'it'},
    {'text': 'Mi piace mangiare la pizza e la pasta.', 'idioma': 'it'},
    {'text': 'Dove si trova la stazione ferroviaria più vicina?', 'idioma': 'it'},
    {'text': 'Questo è un esempio di testo in italiano.', 'idioma': 'it'},
    {'text': 'Il tempo a Roma è bellissimo in primavera.', 'idioma': 'it'},
    {'text': 'Grazie mille per il tuo supporto costante.', 'idioma': 'it'},

    # Francés (fr)
    {'text': 'Bonjour, comment allez-vous ce matin?', 'idioma': 'fr'},
    {'text': "J'aime beaucoup la cuisine française traditionnelle.", 'idioma': 'fr'},
    {'text': "C'est une très belle journée pour une promenade.", 'idioma': 'fr'},
    {'text': 'Le petit déjeuner est prêt dans la cuisine.', 'idioma': 'fr'},
    {'text': 'Pouvez-vous me donner plus de détails sur ce dossier?', 'idioma': 'fr'},
    {'text': "L'apprentissage automatique est passionnant.", 'idioma': 'fr'},

    # Inglés (en)
    {'text': 'Hello, how are you doing today?', 'idioma': 'en'},
    {'text': 'This is a simple sentence to test the model.', 'idioma': 'en'},
    {'text': 'I enjoy reading books about history and science.', 'idioma': 'en'},
    {'text': 'Could you please send me the meeting minutes?', 'idioma': 'en'},
    {'text': 'Machine learning is a subfield of artificial intelligence.', 'idioma': 'en'},
    {'text': 'The weather is quite rainy in London today.', 'idioma': 'en'}
]

df_audit = pd.DataFrame(datos_prueba)

# Preprocesamos
df_audit_proc = preprocesar_dataset(df_audit, config_final)

# Solo procedemos si el modelo y vectorizador existen (asumiendo entrenamiento previo exitoso)
try:
    X_audit_tfidf = vectorizador_tfidf.transform(df_audit_proc['Text_Procesado'])
    df_audit_proc['idioma_predicho'] = mejor_modelo.predict(X_audit_tfidf)

    # Identificamos errores
    errores_audit = df_audit_proc[df_audit_proc['idioma'] != df_audit_proc['idioma_predicho']]

    print(f"Resultados: {len(df_audit_proc) - len(errores_audit)} aciertos de {len(df_audit_proc)} totales.")
    
    if len(errores_audit) > 0:
        print("\nEjemplos donde el modelo falló:")
        display(errores_audit[['text', 'idioma', 'idioma_predicho']])
    else:
        print("\n¡Increíble! El modelo acertó todos los ejemplos del dataset sintético.")
        
    print("\nVista previa de las predicciones (primeros 10):")
    display(df_audit_proc[['text', 'idioma', 'idioma_predicho']].head(10))
    
except NameError:
    print("Error: El modelo o el vectorizador no están definidos. Asegúrate de ejecutar las celdas de entrenamiento primero.")

Resultados: 29 aciertos de 30 totales.

Ejemplos donde el modelo falló:


,text,idioma,idioma_predicho
21,Le petit déjeuner est prêt dans la cuisine.,fr,es



Vista previa de las predicciones (primeros 10):


,text,idioma,idioma_predicho
0,"Hola, ¿cómo estás hoy?",es,es
1,El sistema de detección de idiomas funciona bien.,es,es
2,Mañana iré a caminar por el parque central.,es,es
3,Me gusta mucho la comida mediterránea.,es,es
4,¿Podrías enviarme el reporte final por correo?,es,es
5,La inteligencia artificial está cambiando el m...,es,es
6,"Olá, como você está hoje?",pt,pt
7,Eu gosto muito de viajar para a praia no verão.,pt,pt
8,O café da manhã está servido na mesa.,pt,pt
9,Esta é uma frase curta em português.,pt,pt


Concluimos por tanto que cuando las frases son suficientemente largas, el modelo es bastante bueno. Solo tiene un fallo, la frase ''Le petit déjeuner est prêt dans la cuisine.'' que estima que el idioma es español y es francés.